In [ ]:
# These environment variables are commonly set
# %env LOCAL_DEV_ENV=True
# %env MMM_LICENSE_ACCEPTED=i accept
import os
import uuid
import logfire
logfire.configure(send_to_logfire="if-token-present", sampling=logfire.SamplingOptions.level_or_duration())
import torch
from pathlib import Path
# This one-liner imports MMM utilities relevant to interactive programming.
from mmm.interactive import configs as cfs, data, tasks, training, pipes, blocks, api


In [ ]:
# Most experiments need the same stuff.
# This object reads our standard environment variables and prepares for distributed training if torchrun is used.
env = cfs.EnvByConvention("finetuning").if_torchrun_prepare()

## Config

- By convention, we collect the parameters that should be configurable without code changes into a `HyperParameters` object
  - If your experiment wants to examine the impact of different encoders, it might make sense to include an encoder into the config object
- All objects in the library have config options implemented with pydantic

In [ ]:
from mmm.api.M3Model import UNICORN_ENCODER, M3_MODELS


class HyperParameters(cfs.ExperimentHyperParameters):
    foundation_model: data.DistributedPath | str = M3_MODELS[UNICORN_ENCODER]

    experiment_name: str = "finetuning_example"
    resumable: bool = True  # Whether to resume from an existing checkpoint if available

    trainer: training.MTLTrainer.Config = training.MTLTrainer.Config(
        checkpoint_cache_folder=Path("trainer_checkpoints"),  # by default, config->result=true jobs will be resumed
        train_device="cuda", # "cuda" or "cpu"
        # For experimentation, the steps per loop can be reduced.
        mtl_train_loop=cfs.TrainLoopConfig(max_steps=200),
        mtl_val_loop=cfs.ValLoopConfig(max_steps=100),
        max_epochs=2
    )


In [ ]:
HyperParameters.update_schema(env)

VSCode provides auto-completion for all configuration options via JSON schema. If it does not exist, this command will create `./job_configs/finetuning.jsonc`.

In [ ]:
# In interactive environments the config is loaded from a file that is always located at ./job_configs/env_name.jsonc
config = HyperParameters.load_config(env)

## Anatomy of a multi-task model

- Our multi-task models consist of shared blocks (see `blocks.SharedBlock`) and tasks (see `tasks.MTLTask`)
- All blocks and tasks are PyTorch modules
- We start with 2D classification. A classification task requires
  - a `blocks.PyramidEncoder` which transforms an image Tensor[C, H, W] into feature maps list[Tensor[C, H, W]]
  - a `blocks.Squeezer` which transforms the feature maps list[Tensor[C, H, W]] into a latent representation Tensor[C, H, W]
  - a `tasks.ClassificationTask` which takes a latent representation, make predictions, and visualizes results

In [ ]:
# Download the model into the directory specified by env variable ML_DATA_CACHE, otherwise ~/.mmm/
foundation_model = api.M3Model(config.foundation_model, device_identifier="cuda:0")

# The model can be used as a dictionary of modules.
encoder: blocks.PyramidEncoder = foundation_model["encoder"]
squeezer: blocks.Squeezer = foundation_model["squeezer"]
encoder.freeze_all_parameters()  # freeze the encoder, only other modules (partial fine-tuning)

In [ ]:
# Inputs are batches of images like (batch_size, channels, height, width) which are between 0 and 1.
with torch.no_grad():
    test_input = torch.rand(2, 3, 64, 64).to(encoder.torch_device)
    feature_maps = encoder(test_input)
    latent_feature_map, latent_representation = squeezer(feature_maps)
    print("\n".join([f"{feat_map.shape}" for feat_map in feature_maps]), f"\nLatent: {latent_representation.shape}")

## Logging

For confidential data you should use an internal WANDB instance using `%env WANDB_BASE_URL=http://your-host:PORT/`
For logging to the official servers (including some of your training images by default), create an account at https://wandb.ai/ and be ready to paste your key into here.

The logging is integrated with our config system. All your user-configurable settings should be visible in the overview of the respective experiment:

In [ ]:
wandb_run = config.init_experiment(env)
# Alternatively, W&B can be disabled
# import wandb; wandb.init(mode="disabled")

If everything worked, you should be able to click a link with a randomly generated name for this experiment.
For now, this link should contain:

- your custom config in Workspace->report
- a structured overview of all your config values in Overview->Config

## Preparing data

In this guide, we will start multi-task classification trainings with the https://medmnist.com/ database.
You can install the database using `pip install medmnist`

In [ ]:
try:
    from medmnist import BloodMNIST, OrganMNIST3D
except ImportError:
    %pip install medmnist
    from medmnist import BloodMNIST, OrganMNIST3D
# Folder with enough space for input data:
DATA_ROOT = os.getenv("ML_DATA_CACHE", default="./data")
train_dataset = BloodMNIST(root=DATA_ROOT, split="train", download=True, size=224)
val_dataset = BloodMNIST(root=DATA_ROOT, split="val", download=True, size=224)
class_names = [train_dataset.info['label'][f'{i}'] for i in range(len(train_dataset.info['label']))]
train_dataset[0], train_dataset.info['label'], class_names

MMM uses [PyTorch dataloading](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html) with a specific format. Each new dataset needs to be converted into this dictionary-based format. The MedMNIST dataset has a fixed length. In consequence, we could use a `torch.utils.data.Dataset` to wrap it. However, MedMNIST dataset already support `__len__` and `__getitem__` and can therefore directly be used within MMM.

In [ ]:
import torchvision.transforms.functional as F

def transform_pil_to_mmm(case: tuple):
    """
    MMM datasets are wrappers around PyTorch datasets that require a very specific format for each type of label.
    For classification, a dictionary with "image" and "class" keys is expected.
    """
    pil_image, label = case
    return {
        "image": F.to_tensor(pil_image.convert("RGB")),
        "class": label.item()
    }

# The data is encapsulated in an object that holds a training and a validation set.
def mmm_cohort(train_dataset, val_dataset) -> data.TrainValCohort:
    return data.TrainValCohort(
        data.TrainValCohort.Config(batch_size=(8, 8), num_workers=2), # 8 train, 8 val
        train_ds=data.ClassificationDataset(
            train_dataset,
            src_transform=transform_pil_to_mmm,
            batch_transform=pipes.Alb(pipes.get_weak_default_augs()), # Augmentations are applied only to training.
            class_names=class_names
        ),
        val_ds=data.ClassificationDataset(
            val_dataset,
            src_transform=transform_pil_to_mmm,
            class_names=class_names
        ),
    )
mmm_train_dataset = (my_cohort := mmm_cohort(train_dataset, val_dataset)).datasets[0]
mmm_training_case = mmm_train_dataset[0]
mmm_training_case['image'].shape, mmm_training_case["class"]

Extending to 3D requires to set a "group_id" for each slice. Slices with the same group_id belong to the same 3D volume.

In [ ]:
from mmm.volume3d import Tomo3DProcessor

train_3d = OrganMNIST3D(root=DATA_ROOT, split="train", download=True, size=64)
val_3d = OrganMNIST3D(root=DATA_ROOT, split="val", download=True, size=64)
classes_3d = [train_3d.info['label'][f'{i}'] for i in range(len(train_3d.info['label']))]
train_3d[0][0].shape, train_3d.info['label'], classes_3d

In [ ]:
def transform_volume_to_slices(case: tuple):
    npy_volume, label = case
    assert npy_volume.min()>=0 and npy_volume.max()<=1  # Volume is already normalized for MedMNIST3D
    patient_id = uuid.uuid4().hex[:8]  # Generate a random patient ID for grouping slices
    return [
        {
            # All image inputs are expected to be 3-channel
            "image": Tomo3DProcessor.repeat_channels(torch.from_numpy(npy_volume[..., i])).float(),
            "class": label.item(),
            "meta": {"group_id": patient_id}
        } for i in range(npy_volume.shape[-1])
    ]

def mmm_cohort_3d(train_dataset, val_dataset) -> data.TrainValCohort:
    return data.TrainValCohort(
        data.TrainValCohort.Config(batch_size=(4, 4), num_workers=2), # 4 train, 4 val
        train_ds=data.ClassificationDataset(
            train_dataset,
            src_transform=transform_volume_to_slices,
            batch_transform=pipes.ApplyToList(pipes.Alb(pipes.get_weak_default_augs(), replay_for_groups=True)),
            class_names=classes_3d,
            collate_fn=pipes.mtl_batch_collate
        ),
        val_ds=data.ClassificationDataset(
            val_dataset,
            src_transform=transform_volume_to_slices,
            class_names=classes_3d,
            collate_fn=pipes.mtl_batch_collate
        ),
    )
mmm_train_dataset_3d = (my_cohort := mmm_cohort_3d(train_3d, val_3d)).datasets[0]
mmm_training_case_3d = mmm_train_dataset_3d[0]
len(mmm_training_case_3d), mmm_training_case_3d[0]['image'].shape, mmm_training_case_3d[0]["class"]

## Training your model

`training.MTLTrainer` is responsible for running the multi-task training loop. By default, it uses gradient accumulation to perform update steps consisting of all tasks that were added using `trainer.add_mtl_task(...)`. By default, it starts with a validation loop and runs each loop until exhaustion. The last step of a task might consist of a batch with smaller batchsize than the other steps.

In [ ]:
trainer: training.MTLTrainer = training.MTLTrainer(
    config.trainer,
    experiment_name=cfs.remove_wandb_special_chars(config.experiment_name),
    clear_checkpoints=not config.resumable,
).add_shared_blocks([foundation_model[k] for k in foundation_model.get_sharedblock_keys()])

Each `tasks.MTLTask` assembles its own architecture consisting of its own modules and the shared blocks. In the case of the `tasks.ClassificationTask`, these shared blocks are the shared encoder and the shared squeezer. Each `tasks.MTLTask` needs a unique name.

In [ ]:
mmm_task = tasks.ClassificationTask(
    hidden_dim=squeezer.get_hidden_dim(),
    args=tasks.ClassificationTask.Config(module_name="2dclassification"),
    cohort=my_cohort
)

trainer.add_mtl_task(mmm_task)

In [ ]:
mmm_task_3d = tasks.ClassificationTask(
    hidden_dim=squeezer.get_hidden_dim(),
    # Instruct the task to use the "grouper" transformer module from the foundation model to establish context
    args=tasks.ClassificationTask.Config(
        module_name="3dclassification",
        grouper_key=cfs.GroupUsage(grouper_key="grouper")
    ),
    cohort=mmm_cohort_3d(train_3d, val_3d)
)

trainer.add_mtl_task(mmm_task_3d)

In [ ]:
# After adding all tasks, the trainer is ready for training.
trainer.fit()

## Exporting the model

For different use cases we recommend different export methods:

- Native PyTorch export via `MTLTrainer.save_blocks_native`. This exports an `nn.ModuleDict` object which is expected by our inference utilities. This has the disadvantage that all dependencies have to be installed exactly as they were during the export because this uses `pickle` internally. This method is recommended when you have control over the inference environment (e.g. by using the same container as during training).
- ONNX export via `SharedBlock.export_to_onnx`. This exports a single shared block such as the encoder using the established ONNX standard. This is good for sharing with external users.

In [ ]:
# The trainer exports the modules in their current state. If you want to load a checkpoint first:
# trainer.load_checkpoint(Path("trainer_checkpoints/bigtraining42/bestbyvalidation-3"), load_optim_state=False)
# By default, there should be trainer_checkpoints/[experiment-name]/bestbyvalidation-[epoch] and latest folders.

module_dict = trainer.save_blocks_native(
    export_modules_path := data.DistributedPath.from_string("./all_blocks.pt.zip"),
    only_inference=True  # Cohorts often should not be exported, `only_inference` strips those.
)
module_dict.keys()

In [ ]:
# Loading requires only one line of torch and is not specific to MMM:
exported_dict = api.M3Model(export_modules_path, "cuda:0")
with torch.inference_mode():
    some_feature_maps = exported_dict["encoder"](test_input)
    print(some_feature_maps[-1].shape)

The native PyTorch export requires the user to know how to assemble the blocks together. Alternatively, native export of individual tasks can be used to export a whole task's pipeline including the shared blocks. For this, the `save_task_native` of `MTLTrainer` can be used.

In [ ]:
trainer.save_task_native("unique_task_name", Path("./task.pt"), only_inference=True)

In [ ]:
# Loading again only requires one line and is not specific to MMM:
exported_task = torch.load("./task.pt", weights_only=False)
with torch.inference_mode():
    # The task module returns the logits, torch.argmax is used to get the class index for classification.
    task_output = exported_task.forward(
        (test_input.to(exported_task.task.torch_device), multiple_instance_learning_indices := None)
    )
    # This differs for label types. For example, segmentation tasks have methods for transforming the network output:
    # SemSegTask.logits_to_probas -> SemSegTask.probas_to_preds
    # Useful resources for this are the task's docstring, and `training_step` and visualization methods.
mmm_task.class_names[torch.argmax(task_output)], torch.max(task_output).item()